# What Are Hyperparameters?

**Topic:** Model Configuration & Tuning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Output, HBox, VBox

from IPython.display import display, clear_output

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the difference between model parameters and hyperparameters
- **Explain** why hyperparameter choices determine whether a model generalizes or memorizes
- **Interpret** a training vs. validation accuracy curve to identify the best hyperparameter value

> **Tip:** In the real-world example below, watch what happens to the gap between training and validation accuracy as `max_depth` increases. That gap is overfitting in action.

---
## How we got here

In **[15 — Generalization and the Test Set](15_generalization_and_the_test_set.ipynb)** you learned that a model's job is not just to fit the training data: it has to perform well on data it has never seen before. Once you evaluate on the test set, you cannot use it again.

That raises a practical question. Before you train, how do you decide *how* the model should learn? How deep should the tree grow? How many neighbors should k-NN consider? How fast should gradient descent move?

Those choices are **hyperparameters**, and this notebook is about what they are, why they matter, and how to think about them.

---
## Why this matters for data science

Picking the right algorithm is only part of the job. Two data scientists can use the exact same algorithm on the exact same dataset and get very different results, depending on the hyperparameters they choose.

Hyperparameter tuning is one of the most practical skills in applied machine learning. It is also one of the easiest places to accidentally introduce overfitting: you keep adjusting settings to improve your validation score until the model is essentially tailored to that one held-out slice of data, and then it fails on truly new examples.

Understanding what hyperparameters are, and how they differ from values the model learns on its own, is the foundation you need before tuning anything.

---
## Try it yourself

In [2]:
out = Output()

ALGO_DATA = {
    "Linear Regression": {
        "params": [
            ("coef_", "Slope for each feature — learned from data"),
            ("intercept_", "Baseline prediction when all features are zero — learned from data"),
        ],
        "hyperparams": [
            ("fit_intercept", "Add an intercept term? True or False — you set this before training"),
            ("positive", "Constrain coefficients to be ≥ 0? — you set this before training"),
        ],
    },
    "Decision Tree": {
        "params": [
            ("split thresholds", "Which feature value to divide on at each node — learned from data"),
            ("leaf predictions", "The output value at each terminal node — learned from data"),
        ],
        "hyperparams": [
            ("max_depth", "Maximum depth the tree can grow to — you set this before training"),
            ("min_samples_split", "Fewest examples needed to split a node — you set this before training"),
            ("min_samples_leaf", "Fewest examples required at each leaf — you set this before training"),
        ],
    },
    "K-Nearest Neighbors": {
        "params": [
            ("stored training data", "KNN memorizes all of X_train and y_train — there are no learned weights; the data IS the model"),
        ],
        "hyperparams": [
            ("n_neighbors (k)", "How many neighbors vote on each prediction — you set this before training"),
            ("weights", "Equal votes or weighted by distance — you set this before training"),
            ("metric", "How to measure distance (Euclidean, Manhattan…) — you set this before training"),
        ],
    },
    "Random Forest": {
        "params": [
            ("split thresholds (×n_estimators trees)", "Every split value in every tree — learned from data"),
            ("leaf values (×n_estimators trees)", "Output at every leaf in every tree — learned from data"),
        ],
        "hyperparams": [
            ("n_estimators", "Number of trees to build — you set this before training"),
            ("max_depth", "Maximum depth of each tree — you set this before training"),
            ("max_features", "Features to consider at each split — you set this before training"),
        ],
    },
    "Gradient Boosting": {
        "params": [
            ("split thresholds (×n_estimators trees)", "Split values for each boosting tree — learned from data"),
            ("leaf values (×n_estimators trees)", "Residual corrections at each leaf — learned from data"),
        ],
        "hyperparams": [
            ("n_estimators", "Number of boosting rounds — you set this before training"),
            ("learning_rate", "Contribution weight of each tree — you set this before training"),
            ("max_depth", "Depth of each individual tree — you set this before training"),
        ],
    },
}

algo_dd = Dropdown(
    options=list(ALGO_DATA.keys()),
    value="Decision Tree",
    description="Algorithm:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))

def render(change=None):
    algo = algo_dd.value
    params  = ALGO_DATA[algo]["params"]
    hparams = ALGO_DATA[algo]["hyperparams"]
    n_rows  = max(len(params), len(hparams))
    p_names = [r[0] for r in params]  + [""] * (n_rows - len(params))
    p_descs = [r[1] for r in params]  + [""] * (n_rows - len(params))
    h_names = [r[0] for r in hparams] + [""] * (n_rows - len(hparams))
    h_descs = [r[1] for r in hparams] + [""] * (n_rows - len(hparams))

    fig = go.Figure(go.Table(
        columnwidth=[180, 330, 180, 330],
        header=dict(
            values=[
                "<b>Parameters</b>",
                "<b>What they represent</b> — learned by the model from data",
                "<b>Hyperparameters</b>",
                "<b>What they control</b> — set by you before training",
            ],
            fill_color=[PALETTE["primary"], PALETTE["primary"],
                        PALETTE["secondary"], PALETTE["secondary"]],
            font=dict(color="white", size=12, family=FONT["family"]),
            align="left",
            height=44,
        ),
        cells=dict(
            values=[p_names, p_descs, h_names, h_descs],
            fill_color=[
                ["#eef2ff"] * n_rows,
                ["white"] * n_rows,
                ["#fff3ee"] * n_rows,
                ["white"] * n_rows,
            ],
            font=dict(size=11, family=FONT["family"]),
            align="left",
            height=52,
        ),
    ))
    fig.update_layout(
        title=dict(
            text=f"{algo} — Parameters vs Hyperparameters",
            font=dict(family=FONT["family"], size=16, color="#333"),
        ),
        height=140 + n_rows * 58,
        margin=dict(t=55, b=10, l=10, r=10),
        font=dict(family=FONT["family"]),
        paper_bgcolor="white",
    )
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

algo_dd.observe(render, names="value")
display(VBox([algo_dd, out]))
render()

---
## What's happening?

Every machine learning model involves two distinct kinds of numbers.

**Parameters** are values the model learns automatically during training. You never set them by hand: the training algorithm works them out from the data. For example:
- In **linear regression**, the coefficients (slope and intercept) are parameters.
- In a **decision tree**, the split thresholds — which feature value to divide on at each node — are parameters.

**Hyperparameters** are settings you choose *before* training begins. The model never adjusts them on its own. For example:
- `max_depth` in a decision tree: how deep the tree is allowed to grow
- `learning_rate` in gradient descent: how large each update step is
- `n_estimators` in a random forest: how many trees to build

### Why the distinction matters

If you set `max_depth` too high, the tree memorizes the training data and fails on new examples: overfitting. If you set it too low, the tree is too simple to capture real patterns: underfitting. The model cannot fix either problem on its own. Only you can, by adjusting the hyperparameter.

| | Parameters | Hyperparameters |
|---|---|---|
| Set by | The training algorithm | You, before training |
| Learned from | Data | Not learned: chosen |
| Examples | Coefficients, split thresholds, weights | `max_depth`, `learning_rate`, `n_estimators` |
| Change during training? | Yes | No |

---
## Real-world example: Tuning a Decision Tree

The `max_depth` hyperparameter controls how many levels deep the tree grows. A shallow tree underfits and misses real patterns in the data. A very deep tree overfits and memorizes the training data so thoroughly it loses the ability to generalize.

> **Discussion question:** If training accuracy keeps improving as `max_depth` increases but validation accuracy peaks at depth 5, what does that tell you?

### Common hyperparameters by algorithm

| Algorithm | Key hyperparameters | What they control | Default value |
|-----------|---------------------|-------------------|---------------|
| Decision Tree | `max_depth` | How deep the tree grows | `None` (unlimited) |
| Decision Tree | `min_samples_split` | Minimum samples required to split a node | `2` |
| Random Forest | `n_estimators` | Number of trees in the forest | `100` |
| Random Forest | `max_features` | Number of features considered at each split | `"sqrt"` |
| k-Nearest Neighbors | `n_neighbors` | How many neighbors vote on the prediction | `5` |
| Gradient Boosting | `learning_rate` | Step size for each boosting round | `0.1` |
| Gradient Boosting | `n_estimators` | Number of boosting rounds | `100` |
| Logistic Regression | `C` | Inverse of regularization strength | `1.0` |

---
## Key takeaway

> **Parameters are what the model learns; hyperparameters are what you set — and choosing them wisely is the difference between a model that generalizes and one that memorizes.**

---
*Next up: supervised/ — where you'll tune hyperparameters for every major algorithm*